# Bài 8 v4 · K-means và khách hàng RFM
PGS.TS. Nguyễn Văn Hậu. Chạy từ đầu đến cuối. Sáu điểm là mô phỏng; Online Retail là giao dịch thật. Toàn cohort lịch sử dùng khám phá, không tự chứng minh hiệu quả kinh doanh.

In [1]:
from pathlib import Path
import sys,os,json
ROOT=Path.cwd()
if not (ROOT/"source/kmeans_lab.py").exists():raise RuntimeError("Mở notebook trong thư mục bài đã giải nén")
if (ROOT/".build/pydeps").exists():sys.path.insert(0,str(ROOT/".build/pydeps"))
sys.path.insert(0,str(ROOT/"source"))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from kmeans_lab import P,NAMES,lloyd,run
from sklearn.metrics import silhouette_samples
from threadpoolctl import threadpool_limits
OUT=ROOT/"results/lab_figures";OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({"figure.figsize":(10,5),"axes.grid":True})
print("Cửa sổ lịch sử, tiền GBP, snapshot 10/12/2011")

Cửa sổ lịch sử, tiền GBP, snapshot 10/12/2011


## 1. Sáu điểm và một phép gán
Tọa độ cố định. Tâm đầu là An và Chi. So bình phương khoảng cách từng hàng.

In [2]:
history=lloyd(P,P[[0,2]])
assert np.allclose([z["J"] for z in history],[39,12,93/8,22/3,22/3,22/3])
print(pd.DataFrame({"Điểm":NAMES,"d1^2":np.array(history[0]["d2"])[:,0],"d2^2":np.array(history[0]["d2"])[:,1],"Nhóm":np.array(history[0]["labels"])+1}))
for i,z in enumerate(history):print(i+1,z["phase"],"centers=",z["centers"],"J=",z["J"])

   Điểm  d1^2  d2^2  Nhóm
0    An   0.0   8.0     1
1  Bình   1.0   5.0     1
2   Chi   8.0   0.0     2
3  Dung  25.0   5.0     2
4    Hà  41.0  13.0     2
5   Lan  52.0  20.0     2
1 assign centers= [[1.0, 1.0], [3.0, 3.0]] J= 39.0
2 update centers= [[1.5, 1.0], [5.25, 4.25]] J= 12.0
3 assign centers= [[1.5, 1.0], [5.25, 4.25]] J= 11.625
4 update centers= [[2.0, 1.6666666666666667], [6.0, 4.666666666666667]] J= 7.333333333333333
5 assign centers= [[2.0, 1.6666666666666667], [6.0, 4.666666666666667]] J= 7.333333333333333
6 update centers= [[2.0, 1.6666666666666667], [6.0, 4.666666666666667]] J= 7.333333333333333


## 2. Hai nửa bước thay đổi gì?
Giữ cùng trục và điểm, chỉ nhãn hoặc tâm đổi. Dấu thoi là tâm.

In [3]:
fig,axes=plt.subplots(1,4,figsize=(16,4),sharex=True,sharey=True)
for ax,z in zip(axes,history[:4]):
    lab=np.array(z["labels"]);cent=np.array(z["centers"])
    ax.scatter(P[:,0],P[:,1],c=np.array(["#245a84","#bc761a"])[lab],s=45)
    ax.scatter(cent[:,0],cent[:,1],c=["#245a84","#bc761a"],marker="D",s=100,facecolors="none")
    for name,pnt in zip(NAMES,P):ax.annotate(name,pnt,xytext=(3,6),textcoords="offset points")
    ax.set(xlim=(0,8),ylim=(0,6),xlabel="x1",title=z["phase"]+"; J="+str(round(z["J"],3)));ax.set_aspect("equal")
fig.tight_layout();fig.savefig(OUT/"lloyd_states.png",dpi=180);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_9636\4059310979.py:5: UserWarning: You passed both c and facecolor/facecolors for the markers. c has precedence over facecolor/facecolors. This behavior may change in the future.
  ax.scatter(cent[:,0],cent[:,1],c=["#245a84","#bc761a"],marker="D",s=100,facecolors="none")


C:\Users\nguye\AppData\Local\Temp\ipykernel_9636\4059310979.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/"lloyd_states.png",dpi=180);plt.show()


## 3. Chạy case từ ZIP gốc
Tính RFM rồi log và chuẩn hóa. Kiểm toàn bộ khách bằng ngày Python, tập InvoiceNo và Decimal. K2…6 có cùng ngân sách, mẫu silhouette dùng cùng ID. Ô này có thể mất vài phút.

In [4]:
audit=run()
assert audit["status"]=="pass"
print("Kiểm trong case:",audit["checks_count"],"Khách:",audit["customers"])
print(pd.DataFrame(audit["experiments"])[["k","inertia","silhouette","silhouette_full","sizes"]])

{"status": "pass", "checks_count": 45, "example": {"id": 12755, "lines": 4, "last": "2011-03-04 12:42:00", "R": 281, "F": 2, "M": 3811.95, "raw": [281.0, 2.0, 3811.95], "log": [5.641907070938114, 1.0986122886681098, 8.246158446737232], "scaled": [1.3655993898707732, -0.36158277709840037, 1.314210569745776], "d2_k4": [7.701001423729027, 4.759665026614313, 10.895360326549785, 2.7002972077869254], "cluster_k4": 3}, "experiments": [{"k": 2, "inertia": 6474.146444080157, "silhouette": 0.42523815959757655, "silhouette_subsamples": {"42": 0.42523815959757655, "43": 0.4261952722653358, "44": 0.43689640987064826}, "silhouette_full": 0.43262378510886584, "sizes": [1669, 2669], "iterations": 8}, {"k": 3, "inertia": 4856.484636333079, "silhouette": 0.3300977114094801, "silhouette_subsamples": {"42": 0.3300977114094801, "43": 0.3352802965630655, "44": 0.3335468233606704}, "silhouette_full": null, "sizes": [757, 1886, 1695], "iterations": 10}, {"k": 4, "inertia": 3924.4596992758034, "silhouette": 0.

Kiểm trong case: 45 Khách: 4338
   k      inertia  silhouette  silhouette_full                           sizes
0  2  6474.146444    0.425238         0.432624                    [1669, 2669]
1  3  4856.484636    0.330098              NaN               [757, 1886, 1695]
2  4  3924.459699    0.332372         0.335547          [877, 1579, 720, 1162]
3  5  3268.211919    0.319679              NaN     [1163, 344, 966, 861, 1004]
4  6  2840.151416    0.319438              NaN  [847, 665, 994, 598, 278, 956]


## 4. Theo dấu khách 12755
Bốn dòng, hai InvoiceNo, R ngày lịch. M là mua dương theo recipe.

In [5]:
lines=pd.read_csv(ROOT/"data/example_customer_lines.csv")
rfm=pd.read_csv(ROOT/"data/retail_rfm.csv")
arrays=np.load(ROOT/"data/lab_arrays.npz")
print(lines[["InvoiceNo","InvoiceDate","StockCode","Quantity","UnitPrice","line_value"]].to_string(index=False))
assert lines.InvoiceNo.nunique()==2
assert np.isclose(lines.line_value.sum(),3811.95)
print("RFM và gán:",audit["example"])

 InvoiceNo         InvoiceDate  StockCode  Quantity  UnitPrice  line_value
    537899 2010-12-09 10:44:00      22328      1488       2.55     3794.40
    545645 2011-03-04 12:42:00      22652         1       1.65        1.65
    545645 2011-03-04 12:42:00      22654         1       5.95        5.95
    545645 2011-03-04 12:42:00      22968         1       9.95        9.95
RFM và gán: {'id': 12755, 'lines': 4, 'last': '2011-03-04 12:42:00', 'R': 281, 'F': 2, 'M': 3811.95, 'raw': [281.0, 2.0, 3811.95], 'log': [5.641907070938114, 1.0986122886681098, 8.246158446737232], 'scaled': [1.3655993898707732, -0.36158277709840037, 1.314210569745776], 'd2_k4': [7.701001423729027, 4.759665026614313, 10.895360326549785, 2.7002972077869254], 'cluster_k4': 3}


## 5. Trước và sau log
Histogram dùng toàn cohort. Log đổi thang và hình học; số âm chuẩn hóa không là số hóa đơn âm.

In [6]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(arrays["X0"][:,2],bins=30,color="#245a84");axes[0].set(xlabel="M (GBP)",ylabel="Số khách",title="Gốc")
axes[1].hist(arrays["log"][:,2],bins=30,color="#167a72");axes[1].set(xlabel="ln(1+M)",ylabel="Số khách",title="Cùng khách sau log")
fig.tight_layout();fig.savefig(OUT/"money_before_after.png",dpi=180);plt.show()
print(pd.DataFrame({"r":audit["example"]["raw"],"u":audit["example"]["log"],"x":audit["example"]["scaled"]},index=["R","F","M"]))

         r         u         x
R   281.00  5.641907  1.365599
F     2.00  1.098612 -0.361583
M  3811.95  8.246158  1.314211


C:\Users\nguye\AppData\Local\Temp\ipykernel_9636\2012218193.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/"money_before_after.png",dpi=180);plt.show()


## 6. Silhouette tính tay
Euclid giữa các điểm. Với điểm 2, a=2 và b=9. Không bình phương và không thay bằng khoảng cách đến tâm.

In [7]:
ss=silhouette_samples(np.array([[0.],[2.],[10.],[12.]]),[0,0,1,1])
assert np.isclose(ss[1],7/9)
print("Silhouette:",ss,"tại điểm 2:",ss[1])
print("a=2,b=1:",(1-2)/max(2,1))
print("Mẫu và toàn cohort K2/K4:",[(z["k"],z["silhouette"],z["silhouette_full"]) for z in audit["experiments"] if z["k"] in [2,4]])

Silhouette: [0.81818182 0.77777778 0.77777778 0.81818182] tại điểm 2: 0.7777777777777778
a=2,b=1: -0.5
Mẫu và toàn cohort K2/K4: [(2, 0.42523815959757655, 0.43262378510886584), (4, 0.3323721293344987, 0.3355470584455458)]


## 7. Đọc K và hồ sơ
Không xếp các biểu diễn khác nhau chỉ theo S. Hồ sơ dưới đây là trung vị RFM gốc.

In [8]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
ks=[z["k"] for z in audit["experiments"]]
axes[0].plot(ks,[z["inertia"] for z in audit["experiments"]],"o-",color="#245a84");axes[0].set(xlabel="K",ylabel="J toàn cohort")
axes[1].plot(ks,[z["silhouette"] for z in audit["experiments"]],"o-",color="#167a72");axes[1].set(xlabel="K",ylabel="S trong mẫu 2000 seed42")
fig.tight_layout();fig.savefig(OUT/"K_evidence.png",dpi=180);plt.show()
print(pd.DataFrame(audit["profile"]))
print("Tâm đổi về gốc:",np.array(audit["inverse_centers_k4"]))

   cluster_k4     n      R     F        M
0           0   877   19.0   2.0   450.84
1           1  1579  183.0   1.0   300.42
2           2   720    9.0  10.0  3726.08
3           3  1162   54.0   4.0  1353.74
Tâm đổi về gốc: [[1.58707451e+01 1.88991053e+00 4.22536028e+02]
 [1.57886090e+02 1.27025973e+00 2.71711512e+02]
 [8.26517505e+00 1.08764479e+01 4.22443601e+03]
 [5.31415905e+01 3.84378646e+00 1.42274297e+03]]


C:\Users\nguye\AppData\Local\Temp\ipykernel_9636\2366971004.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/"K_evidence.png",dpi=180);plt.show()


## 8. Kiểm độ nhạy
Mỗi seed có n_init20 như mô hình tham chiếu. Bỏ trùng refit toàn pipeline; ARI không cần cùng tên nhãn.

In [9]:
print(pd.DataFrame(audit["stability"]))
print("Ngân sách:",audit["stability_budget"])
print("ARI bỏ trùng:",audit["dedup"]["ARI"])
print(pd.DataFrame(audit["dedup"]["aligned_transition"],index=["Giữ:N0","Giữ:N1","Giữ:N2","Giữ:N3"],columns=["Bỏ:N0","Bỏ:N1","Bỏ:N2","Bỏ:N3"]))
assert np.array(audit["dedup"]["aligned_transition"]).sum()==4338
print(pd.DataFrame(audit["dedup"]["aligned_profiles"]))

    seed      inertia       ARI
0      0  3924.725449  0.923690
1      1  3924.470998  0.999342
2      2  3924.738057  0.919446
3      3  3924.833921  0.933532
4      4  3924.660760  0.952615
5      5  3924.700323  0.946684
6      6  3924.691353  0.948488
7      7  3924.845292  0.915465
8      8  3924.470889  0.997327
9      9  3924.499659  0.996166
10    10  3924.473238  1.000000
11    11  3924.694557  0.952994
Ngân sách: {'n_init': 20, 'seeds': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 'reference_seed': 42, 'reference_n_init': 20}
ARI bỏ trùng: 0.9575647564735956
        Bỏ:N0  Bỏ:N1  Bỏ:N2  Bỏ:N3
Giữ:N0    850      0      8     19
Giữ:N1     39   1539      0      1
Giữ:N2      0      0    720      0
Giữ:N3      0      2      3   1157
   aligned_cluster     n      R     F        M
0                0   889   20.0   2.0   418.40
1                1  1541  186.0   1.0   300.95
2                2   731    9.0  10.0  3653.75
3                3  1177   54.0   4.0  1337.72


## 9. Tự đánh giá sản phẩm
Chạy kiểm toán độc lập rồi viết một mục đích, hai bằng chứng, một giới hạn và một phép kiểm tiếp. K là số cụm, không là số tọa độ.

In [10]:
import runpy
runpy.run_path(str(ROOT/"source/science_audit.py"),run_name="__main__")
assert json.loads((ROOT/"results/science_audit.json").read_text(encoding="utf8"))["status"]=="pass"
print("Gợi ý: K2 có S cao hơn; K4 có nhiều hồ sơ hơn. Chưa có dữ liệu hiệu quả hành động.")
print("Tự học: thay từng K/seed/recipe, giữ và báo cáo điều kiện so sánh.")

{"status": "pass", "checks_count": 117}
Gợi ý: K2 có S cao hơn; K4 có nhiều hồ sơ hơn. Chưa có dữ liệu hiệu quả hành động.
Tự học: thay từng K/seed/recipe, giữ và báo cáo điều kiện so sánh.
